# Data Engineering

In [1]:
import os
import time
import pickle
import shutil
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import median_filter

# ==========================================
# 1. KONFIGURASI DIREKTORI KAGGLE
# ==========================================
DATA_DIR = '/kaggle/input/competitions/ariel-data-challenge-2025'

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"Dataset tidak ditemukan di {DATA_DIR}. Pastikan Anda telah menekan 'Add Data'.")

print(f"Menggunakan direktori dataset: {DATA_DIR}")

# ==========================================
# 2. FUNGSI PRAPEMROSESAN UTAMA
# ==========================================
def adc_conversion(data, instrument, adc_info_df):
    cols = adc_info_df.columns.tolist()
    
    gain_col_name = f'{instrument}_adc_gain'
    offset_col_name = f'{instrument}_adc_offset'
    
    if gain_col_name in cols and offset_col_name in cols:
        gain = adc_info_df[gain_col_name].values[0]
        offset = adc_info_df[offset_col_name].values[0]
    elif 'sensor' in cols:
        gain = adc_info_df[adc_info_df['sensor'] == instrument]['gain'].values[0]
        offset = adc_info_df[adc_info_df['sensor'] == instrument]['offset'].values[0]
    elif 'instrument' in cols:
        gain = adc_info_df[adc_info_df['instrument'] == instrument]['gain'].values[0]
        offset = adc_info_df[adc_info_df['instrument'] == instrument]['offset'].values[0]
    else:
        raise KeyError(f"Struktur adc_info.csv tidak dikenali. Kolom: {cols}")

    return (data.astype(np.float64) / gain) + offset

def remove_cosmic_rays(data_cube, threshold=5.0, size=3):
    median_filtered = median_filter(data_cube, size=(size, 1, 1))
    diff = np.abs(data_cube - median_filtered)
    mad = np.median(diff) + 1e-6
    outliers = diff > (threshold * mad)
    return np.where(outliers, median_filtered, data_cube)

def process_single_observation(planet_id, obs_idx, adc_info_df, fgs_bin_size, is_train=True):
    split = 'train' if is_train else 'test'
    base_path = os.path.join(DATA_DIR, split, str(planet_id))
    
    result = {'planet_id': planet_id, 'obs_idx': obs_idx, 'fgs_signal': None, 'airs_signal': None}
    
    # --- PROSES FGS1 ---
    fgs_path = os.path.join(base_path, f'FGS1_signal_{obs_idx}.parquet')
    if os.path.exists(fgs_path):
        fgs_raw = pq.read_table(fgs_path).to_pandas().values.reshape(135000, 32, 32)
        fgs_photons = adc_conversion(fgs_raw, 'FGS1', adc_info_df)
        fgs_clean = remove_cosmic_rays(fgs_photons)
        
        # Binning dinamis mengikuti ukuran dari Matriks RBSL
        fgs_time_binned = fgs_clean.reshape(135000 // fgs_bin_size, fgs_bin_size, 32, 32).sum(axis=1)
        result['fgs_signal'] = fgs_time_binned.sum(axis=(1, 2))
        
    # --- PROSES AIRS-CH0 ---
    airs_path = os.path.join(base_path, f'AIRS-CH0_signal_{obs_idx}.parquet')
    if os.path.exists(airs_path):
        airs_raw = pq.read_table(airs_path).to_pandas().values.reshape(11250, 32, 356)
        airs_photons = adc_conversion(airs_raw, 'AIRS-CH0', adc_info_df)
        airs_clean = remove_cosmic_rays(airs_photons)
        
        # Binning AIRS tetap konstan (5) agar selaras dengan baseline astrofisika
        airs_time_binned = airs_clean.reshape(11250 // 5, 5, 32, 356).sum(axis=1)
        result['airs_signal'] = airs_time_binned.sum(axis=1)
        
    return result

# ==========================================
# 3. PEMROSESAN MASSAL UNTUK MATRIKS RBSL
# ==========================================
def process_and_save_all(meta_df, adc_info_df, fgs_bin_size, is_train=True):
    split_name = 'train' if is_train else 'test'
    save_dir = f'/kaggle/working/processed_data_bin_{fgs_bin_size}'
    
    # Hapus data lama jika ada agar bersih
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"\nMemulai pemrosesan dataset {split_name.upper()} dengan Binning FGS: {fgs_bin_size}")
    status_log = []
    
    # Dibatasi 5 planet pertama dulu untuk uji coba (hapus [:5] nanti untuk full dataset)
    for index, row in meta_df[:5].iterrows():
        planet_id = int(row['planet_id']) 
        
        print(f"[{index+1}/5] Memproses Planet ID: {planet_id}...", end=" ")
        start_time = time.time()
        
        try:
            processed_data = process_single_observation(
                planet_id=planet_id, 
                obs_idx=0, 
                adc_info_df=adc_info_df,
                fgs_bin_size=fgs_bin_size,
                is_train=is_train
            )
            
            save_path = os.path.join(save_dir, f"{split_name}_{planet_id}.pkl")
            with open(save_path, 'wb') as f:
                pickle.dump(processed_data, f)
                
            if processed_data['airs_signal'] is not None:
                print(f"Selesai ({time.time() - start_time:.1f}s) - Data Valid")
            else:
                print(f"Selesai ({time.time() - start_time:.1f}s) - Data KOSONG")
                
            status_log.append({'planet_id': planet_id, 'status': 'Success'})
            
        except Exception as e:
            print(f"Gagal. Error: {str(e)}")
            status_log.append({'planet_id': planet_id, 'status': 'Failed'})
            
    pd.DataFrame(status_log).to_csv(os.path.join(save_dir, f"{split_name}_processing_log.csv"), index=False)
    print(f"Data Binning {fgs_bin_size} disimpan di {save_dir}")

# ==========================================
# 4. BLOK EKSEKUSI UTAMA
# ==========================================
print("\nMemuat Metadata...")
adc_info = pd.read_csv(os.path.join(DATA_DIR, 'adc_info.csv'))
train_meta = pd.read_csv(os.path.join(DATA_DIR, 'train_star_info.csv'))

# Menjalankan loop pemrosesan untuk ketiga ukuran time binning
binning_list = [25, 50, 75]
for bin_size in binning_list:
    process_and_save_all(train_meta, adc_info, fgs_bin_size=bin_size, is_train=True)

print("\nSELURUH TAHAP 1 SELESAI. Semua folder data siap digunakan untuk Matriks RBSL.")

Menggunakan direktori dataset: /kaggle/input/competitions/ariel-data-challenge-2025

Memuat Metadata...

Memulai pemrosesan dataset TRAIN dengan Binning FGS: 25
[1/5] Memproses Planet ID: 34983... Selesai (25.2s) - Data Valid
[2/5] Memproses Planet ID: 1873185... Selesai (23.7s) - Data Valid
[3/5] Memproses Planet ID: 3849793... Selesai (23.6s) - Data Valid
[4/5] Memproses Planet ID: 8456603... Selesai (23.6s) - Data Valid
[5/5] Memproses Planet ID: 23615382... Selesai (24.1s) - Data Valid
Data Binning 25 disimpan di /kaggle/working/processed_data_bin_25

Memulai pemrosesan dataset TRAIN dengan Binning FGS: 50
[1/5] Memproses Planet ID: 34983... Selesai (22.6s) - Data Valid
[2/5] Memproses Planet ID: 1873185... Selesai (22.7s) - Data Valid
[3/5] Memproses Planet ID: 3849793... Selesai (22.8s) - Data Valid
[4/5] Memproses Planet ID: 8456603... Selesai (22.7s) - Data Valid
[5/5] Memproses Planet ID: 23615382... Selesai (23.4s) - Data Valid
Data Binning 50 disimpan di /kaggle/working/proc

# Matriks RBSL

In [2]:
import pandas as pd
import os

# ==========================================
# PEMBUATAN JADWAL EKSPERIMEN RBSL 3x3
# ==========================================

def generate_rbsl_matrix():
    # Definisi parameter eksperimen
    iterasi_list = [4, 8, 12]
    binning_list = [25, 50, 75]
    
    # Pola Latin Square 3x3 standar
    # Indeks: 0=A (PCA), 1=B (GP), 2=C (GP+Fudging)
    latin_square_pattern = [
        ['A', 'B', 'C'],
        ['B', 'C', 'A'],
        ['C', 'A', 'B']
    ]
    
    # Kamus terjemahan kode perlakuan
    model_dict = {
        'A': 'PCA_Only',
        'B': 'Gaussian_Process',
        'C': 'GP_and_Fudging'
    }
    
    experiment_runs = []
    run_id = 1
    
    # Menyusun kombinasi menjadi daftar baris
    for i, iterasi in enumerate(iterasi_list):
        for j, binning in enumerate(binning_list):
            kode_perlakuan = latin_square_pattern[i][j]
            nama_model = model_dict[kode_perlakuan]
            
            # Menentukan nama folder data spesifik untuk tiap ukuran binning
            nama_folder_data = f"processed_data_bin_{binning}"
            
            experiment_runs.append({
                'run_id': run_id,
                'iterasi_solver': iterasi,
                'time_binning': binning,
                'kode_perlakuan': kode_perlakuan,
                'nama_model': nama_model,
                'data_folder': nama_folder_data,  # Kolom navigasi folder untuk solver
                'gll_score': None 
            })
            run_id += 1
            
    return pd.DataFrame(experiment_runs)

# Eksekusi dan tampilkan jadwal
rbsl_schedule = generate_rbsl_matrix()

print("=== JADWAL MASTER EKSPERIMEN RBSL ===")
print(rbsl_schedule.to_string(index=False))

# Simpan di root direktori kerja
ROOT_DIR = '/kaggle/working'
schedule_path = os.path.join(ROOT_DIR, 'rbsl_schedule_master.csv')
rbsl_schedule.to_csv(schedule_path, index=False)

print(f"\nJadwal master telah disimpan secara aman di: {schedule_path}")

=== JADWAL MASTER EKSPERIMEN RBSL ===
 run_id  iterasi_solver  time_binning kode_perlakuan       nama_model           data_folder gll_score
      1               4            25              A         PCA_Only processed_data_bin_25      None
      2               4            50              B Gaussian_Process processed_data_bin_50      None
      3               4            75              C   GP_and_Fudging processed_data_bin_75      None
      4               8            25              B Gaussian_Process processed_data_bin_25      None
      5               8            50              C   GP_and_Fudging processed_data_bin_50      None
      6               8            75              A         PCA_Only processed_data_bin_75      None
      7              12            25              C   GP_and_Fudging processed_data_bin_25      None
      8              12            50              A         PCA_Only processed_data_bin_50      None
      9              12            75       

# Konstruksi Core Solver (Bayesian Inference)

In [3]:
!pip install batman-package

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 5.4 MB/s eta 0:00:00


In [4]:
import os
import time
import pickle
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import batman

# ==========================================
# 1. KONFIGURASI DIREKTORI JADWAL MASTER
# ==========================================
ROOT_DIR = '/kaggle/working'
SCHEDULE_PATH = os.path.join(ROOT_DIR, 'rbsl_schedule_master.csv')

if not os.path.exists(SCHEDULE_PATH):
    raise FileNotFoundError("Jadwal RBSL master tidak ditemukan. Pastikan Tahap 2 sudah dijalankan.")

rbsl_schedule = pd.read_csv(SCHEDULE_PATH)
print("Memuat Jadwal Eksperimen RBSL...")

# ==========================================
# 2. KELAS BAYESIAN SOLVER
# ==========================================
class BayesianTransitSolver:
    def __init__(self, time_array, flux_array, model_type):
        self.t = time_array
        self.flux = flux_array
        self.model_type = model_type
        
        # Inisialisasi Prior (Asumsi awal parameter fisika Kepler)
        self.params = batman.TransitParams()
        self.params.t0 = 0.0                        
        self.params.per = 1.0                       
        self.params.rp = 0.1                        
        self.params.a = 15.0                        
        self.params.inc = 87.0                      
        self.params.ecc = 0.0                       
        self.params.w = 90.0                        
        self.params.u = [0.1, 0.3]                  
        self.params.limb_dark = "quadratic"
        
        self.model = batman.TransitModel(self.params, self.t)
        
    def _negative_log_likelihood(self, theta):
        """Menghitung seberapa jauh tebakan dari data nyata (GLL Loss)"""
        self.params.rp = theta[0]
        self.params.t0 = theta[1]
        
        flux_model = self.model.light_curve(self.params)
        residual = self.flux - flux_model
        
        # Penyesuaian batas toleransi noise (sigma) berdasarkan metode Juara 1
        sigma = 0.005 
        if self.model_type == 'Gaussian_Process':
            sigma = 0.003
        elif self.model_type == 'GP_and_Fudging':
            sigma = 0.001
            
        # Rumus GLL (Gaussian Log-Likelihood)
        nll = 0.5 * np.sum((residual / sigma)**2 + np.log(2 * np.pi * sigma**2))
        return nll

    def fit(self, max_iterations):
        """Mencari nilai optimal menggunakan iterasi non-linier L-BFGS-B"""
        initial_guess = [0.1, 0.0] 
        bounds = ((0.001, 0.5), (-0.1, 0.1))
        
        result = minimize(
            self._negative_log_likelihood, 
            initial_guess, 
            method='L-BFGS-B', 
            bounds=bounds,
            options={'maxiter': max_iterations, 'disp': False}
        )
        
        estimated_rp = result.x[0]
        transit_depth = estimated_rp ** 2  # Kedalaman transit adalah kuadrat dari radius
        return transit_depth, result.fun

# ==========================================
# 3. FUNGSI PENCARI DATA DINAMIS
# ==========================================
def get_valid_signal(folder_path):
    """Mencari 1 file .pkl yang valid dari folder binning spesifik"""
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder {folder_path} belum ada. Anda harus menjalankan Tahap 1 untuk binning ini terlebih dahulu.")
        
    data_files = [f for f in os.listdir(folder_path) if f.endswith('.pkl')]
    
    for filename in data_files:
        filepath = os.path.join(folder_path, filename)
        try:
            with open(filepath, 'rb') as f:
                temp_data = pickle.load(f)
            
            if temp_data.get('airs_signal') is not None:
                # Proses sinyal jadi 1D
                if temp_data['airs_signal'].ndim == 2:
                    sig_1d = np.mean(temp_data['airs_signal'], axis=1)
                else:
                    sig_1d = temp_data['airs_signal']
                
                time_arr = np.linspace(-0.05, 0.05, len(sig_1d))
                norm_flux = sig_1d / np.median(sig_1d)
                
                return time_arr, norm_flux, filename
        except Exception:
            continue
            
    raise ValueError(f"Tidak ada data valid yang bisa diproses di dalam {folder_path}.")

# ==========================================
# 4. SIKLUS EKSEKUSI (TRAINING LOOP RBSL)
# ==========================================
print("\nMemulai 9 Skenario Eksperimen RBSL...\n")

for index, run in rbsl_schedule.iterrows():
    run_id = run['run_id']
    iterasi = run['iterasi_solver']
    model_name = run['nama_model']
    data_folder_name = run['data_folder']
    
    print(f"-> Run {run_id}/9 | Model: {model_name} | Iterasi: {iterasi} | Folder: {data_folder_name}")
    
    start_time = time.time()
    
    # Ambil data spesifik sesuai ukuran binning di jadwal
    target_folder = os.path.join(ROOT_DIR, data_folder_name)
    dummy_time, airs_normalized, used_file = get_valid_signal(target_folder)
    
    # Inisialisasi dan eksekusi solver
    solver = BayesianTransitSolver(
        time_array=dummy_time, 
        flux_array=airs_normalized, 
        model_type=model_name
    )
    
    transit_depth, final_gll = solver.fit(max_iterations=iterasi)
    elapsed = time.time() - start_time
    
    print(f"   [Data: {used_file}] Selesai [{elapsed:.2f}s] | GLL Loss: {final_gll:.2f}")
    
    # Simpan hasil loss ke dalam tabel jadwal
    rbsl_schedule.at[index, 'gll_score'] = final_gll

# ==========================================
# 5. PENYIMPANAN HASIL
# ==========================================
rbsl_schedule.to_csv(SCHEDULE_PATH, index=False)
print("\n=== SELURUH EKSPERIMEN SELESAI ===")
print(f"Hasil GLL Score telah disimpan ke {SCHEDULE_PATH}")

Memuat Jadwal Eksperimen RBSL...

Memulai 9 Skenario Eksperimen RBSL...

-> Run 1/9 | Model: PCA_Only | Iterasi: 4 | Folder: processed_data_bin_25
   [Data: train_1873185.pkl] Selesai [0.01s] | GLL Loss: 1240651.06
-> Run 2/9 | Model: Gaussian_Process | Iterasi: 4 | Folder: processed_data_bin_50
   [Data: train_1873185.pkl] Selesai [0.01s] | GLL Loss: 3462621.11
-> Run 3/9 | Model: GP_and_Fudging | Iterasi: 4 | Folder: processed_data_bin_75
   [Data: train_1873185.pkl] Selesai [0.00s] | GLL Loss: 31249141.75
-> Run 4/9 | Model: Gaussian_Process | Iterasi: 8 | Folder: processed_data_bin_25
   [Data: train_1873185.pkl] Selesai [0.00s] | GLL Loss: 3462621.11
-> Run 5/9 | Model: GP_and_Fudging | Iterasi: 8 | Folder: processed_data_bin_50
   [Data: train_1873185.pkl] Selesai [0.00s] | GLL Loss: 31249141.75
-> Run 6/9 | Model: PCA_Only | Iterasi: 8 | Folder: processed_data_bin_75
   [Data: train_1873185.pkl] Selesai [0.01s] | GLL Loss: 1240651.06
-> Run 7/9 | Model: GP_and_Fudging | Iterasi:

/tmp/ipykernel_16/3236440309.py:68: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = minimize(


# Pelatihan & Optimasi "Fudging"

In [5]:
!pip install scikit-learn

In [6]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ==========================================
# 1. KONFIGURASI DIREKTORI
# ==========================================
DATA_DIR = '/kaggle/input/competitions/ariel-data-challenge-2025'
ROOT_DIR = '/kaggle/working'
SCHEDULE_PATH = os.path.join(ROOT_DIR, 'rbsl_schedule_master.csv')

print("Memuat data Ground Truth dan jadwal eksperimen...")
# Memuat ground truth (target sebenarnya yang disembunyikan penyelenggara di data latih)
ground_truth_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
star_info_df = pd.read_csv(os.path.join(DATA_DIR, 'train_star_info.csv'))

# ==========================================
# 2. KELAS FUDGING OPTIMIZER
# ==========================================
class FudgingOptimizer:
    def __init__(self, alpha=1.0):
        """
        Menggunakan Ridge Regression. 
        Alpha adalah penalti (regularization) agar fudging tidak overfitting.
        Terlalu banyak fudging akan menghancurkan skor di Hidden Test Set Kaggle.
        """
        self.model = Ridge(alpha=alpha)
        self.is_trained = False
        
    def prepare_features(self, bayes_predictions, star_metadata):
        """
        Menggabungkan prediksi fisika dengan metadata bintang sebagai fitur.
        Di dunia nyata, jcottaar menggunakan 12 parameter koreksi.
        Di sini kita mensimulasikannya menggunakan parameter bintang (Ts, Rs, dll).
        """
        # Gabungkan prediksi bayes dengan metadata bintang
        features = pd.merge(bayes_predictions, star_metadata, on='planet_id', how='left')
        
        # Pilih kolom fitur yang akan digunakan untuk mengoreksi bias
        # Misalnya: suhu bintang (Ts), radius bintang (Rs), periode (P)
        X = features[['bayes_depth', 'Ts', 'Rs', 'P']].fillna(0)
        return X

    def train(self, bayes_predictions, star_metadata, ground_truth):
        print("Mempersiapkan data pelatihan Fudging...")
        
        # Selaraskan data prediksi dengan ground truth
        merged_data = pd.merge(bayes_predictions, ground_truth, on='planet_id', how='inner')
        
        # Dalam kompetisi ini, ground truth biasanya adalah spektrum multi-wavelength.
        # Untuk penyederhanaan pipeline, kita ambil rata-rata kedalaman transit target.
        # Asumsi kolom target dimulai dari indeks ke-1 (setelah planet_id)
        target_columns = [col for col in ground_truth.columns if col != 'planet_id']
        y_target = merged_data[target_columns].mean(axis=1) 
        
        # Ekstrak matriks fitur X
        X_train = self.prepare_features(merged_data[['planet_id', 'bayes_depth']], star_metadata)
        
        print("Melatih bobot koreksi (Fudge Factors)...")
        self.model.fit(X_train, y_target)
        self.is_trained = True
        
        # Evaluasi internal (Training Error)
        y_pred = self.model.predict(X_train)
        mse = mean_squared_error(y_target, y_pred)
        print(f"Pelatihan Selesai. Mean Squared Error (MSE): {mse:.7f}")
        
        return self.model

    def apply_fudging(self, bayes_predictions, star_metadata):
        if not self.is_trained:
            raise Exception("Model Fudging belum dilatih!")
            
        X_test = self.prepare_features(bayes_predictions, star_metadata)
        corrected_depth = self.model.predict(X_test)
        return corrected_depth

# ==========================================
# 3. SIMULASI & EKSEKUSI PELATIHAN FUDGING
# ==========================================
# Karena di Tahap 3 kita memproses 1 planet per baris RBSL, 
# kita akan membuat dataframe dummy yang berisi kumpulan hasil Bayesian solver 
# agar regresi memiliki cukup data untuk dilatih.

print("\n--- Mengeksekusi Modul Fudging ---")

# Simulasi data output dari Tahap 3 (Bayesian Solver) untuk 50 planet
simulated_planet_ids = star_info_df['planet_id'].head(50).values
# Simulasi: prediksi bayes biasanya memiliki error/bias alami (misal tebakan + sedikit noise)
simulated_bayes_depths = np.random.normal(loc=0.005, scale=0.001, size=50) 

bayes_output_df = pd.DataFrame({
    'planet_id': simulated_planet_ids,
    'bayes_depth': simulated_bayes_depths
})

# Inisiasi dan latih Fudging Optimizer
fudger = FudgingOptimizer(alpha=0.5)
fudger.train(
    bayes_predictions=bayes_output_df,
    star_metadata=star_info_df,
    ground_truth=ground_truth_df
)

# Terapkan koreksi pada data simulasi
final_corrected_depths = fudger.apply_fudging(bayes_output_df, star_info_df)

print("\nPerbandingan Hasil (5 Planet Pertama):")
for i in range(5):
    pid = simulated_planet_ids[i]
    bayes_val = simulated_bayes_depths[i]
    fudge_val = final_corrected_depths[i]
    print(f"Planet {pid} | Prediksi Fisika Murni: {bayes_val:.6f} --> Setelah Fudging: {fudge_val:.6f}")

print("\nModul Fudging siap diintegrasikan ke dalam pipeline submission akhir.")

Memuat data Ground Truth dan jadwal eksperimen...

--- Mengeksekusi Modul Fudging ---
Mempersiapkan data pelatihan Fudging...
Melatih bobot koreksi (Fudge Factors)...
Pelatihan Selesai. Mean Squared Error (MSE): 0.0000461

Perbandingan Hasil (5 Planet Pertama):
Planet 34983 | Prediksi Fisika Murni: 0.005182 --> Setelah Fudging: 0.017922
Planet 1873185 | Prediksi Fisika Murni: 0.003929 --> Setelah Fudging: 0.008221
Planet 3849793 | Prediksi Fisika Murni: 0.005089 --> Setelah Fudging: 0.026012
Planet 8456603 | Prediksi Fisika Murni: 0.006623 --> Setelah Fudging: 0.014548
Planet 23615382 | Prediksi Fisika Murni: 0.003991 --> Setelah Fudging: 0.012268

Modul Fudging siap diintegrasikan ke dalam pipeline submission akhir.


# Evaluasi Statistik (ANOVA)

In [7]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import os

# ==========================================
# 1. MEMUAT DATA HASIL EKSPERIMEN
# ==========================================
ROOT_DIR = '/kaggle/working'
SCHEDULE_PATH = os.path.join(ROOT_DIR, 'rbsl_schedule_master.csv')

if not os.path.exists(SCHEDULE_PATH):
    raise FileNotFoundError("File rbsl_schedule_master.csv tidak ditemukan.")

df = pd.read_csv(SCHEDULE_PATH)

# Validasi kelengkapan data
if df['gll_score'].isnull().any():
    raise ValueError("Eksperimen belum selesai! Ada nilai GLL Score yang masih kosong (None). Pastikan Tahap 3 sukses untuk ke-9 skenario.")

print("Memproses Analisis Ragam (ANOVA) untuk Matriks RBSL 3x3...\n")

# ==========================================
# 2. PERHITUNGAN JUMLAH KUADRAT (JK)
# ==========================================
r = 3  # Ukuran matriks 3x3
N = r * r

# Faktor Koreksi (FK)
G = df['gll_score'].sum()
FK = (G**2) / N

# JK Total
JKT = (df['gll_score']**2).sum() - FK

# JK Baris (Iterasi Solver)
R = df.groupby('iterasi_solver')['gll_score'].sum()
JKB = (R**2).sum() / r - FK

# JK Kolom (Time Binning)
C = df.groupby('time_binning')['gll_score'].sum()
JKK = (C**2).sum() / r - FK

# JK Perlakuan (Model Type)
T = df.groupby('nama_model')['gll_score'].sum()
JKP = (T**2).sum() / r - FK

# JK Sisaan (Error)
JKS = JKT - JKB - JKK - JKP

# ==========================================
# 3. DERAJAT BEBAS (DB) & KUADRAT TENGAH (KT)
# ==========================================
db_B = r - 1
db_K = r - 1
db_P = r - 1
db_S = (r - 1) * (r - 2)
db_T = N - 1

KTB = JKB / db_B
KTK = JKK / db_K
KTP = JKP / db_P
KTS = JKS / db_S

# ==========================================
# 4. UJI F (F-HITUNG & P-VALUE)
# ==========================================
# Mencegah pembagian dengan nol jika error sangat kecil
epsilon = 1e-10
KTS_safe = KTS if KTS > epsilon else epsilon

F_B = KTB / KTS_safe
F_K = KTK / KTS_safe
F_P = KTP / KTS_safe

# Menghitung P-Value
p_B = 1 - stats.f.cdf(F_B, db_B, db_S)
p_K = 1 - stats.f.cdf(F_K, db_K, db_S)
p_P = 1 - stats.f.cdf(F_P, db_P, db_S)

# Nilai F-Tabel pada taraf nyata (alpha) 5%
F_tabel_05 = stats.f.ppf(1 - 0.05, db_B, db_S)

# ==========================================
# 5. MENCETAK TABEL ANOVA
# ==========================================
print("-" * 80)
print(f"{'SUMBER KERAGAMAN':<20} | {'DB':<3} | {'JUMLAH KUADRAT (JK)':<20} | {'KUADRAT TENGAH (KT)':<20} | {'F-HITUNG':<10} | {'P-VALUE'}")
print("-" * 80)
print(f"{'Baris (Iterasi)':<20} | {db_B:<3} | {JKB:<20.4f} | {KTB:<20.4f} | {F_B:<10.4f} | {p_B:.4f}")
print(f"{'Kolom (Binning)':<20} | {db_K:<3} | {JKK:<20.4f} | {KTK:<20.4f} | {F_K:<10.4f} | {p_K:.4f}")
print(f"{'Perlakuan (Model)':<20} | {db_P:<3} | {JKP:<20.4f} | {KTP:<20.4f} | {F_P:<10.4f} | {p_P:.4f}")
print(f"{'Sisaan (Error)':<20} | {db_S:<3} | {JKS:<20.4f} | {KTS:<20.4f} | {'-':<10} | -")
print("-" * 80)
print(f"{'Total':<20} | {db_T:<3} | {JKT:<20.4f}")
print("-" * 80)

# ==========================================
# 6. KESIMPULAN OTOMATIS
# ==========================================
print(f"\nF-Tabel (alpha=0.05): {F_tabel_05:.4f}\n")
print("KESIMPULAN PENELITIAN:")

def ambil_keputusan(f_hitung, f_tabel, nama_faktor):
    if f_hitung > f_tabel:
        return f"Signifikan (H1 Diterima): {nama_faktor} memberikan pengaruh nyata terhadap GLL Loss."
    else:
        return f"Tidak Signifikan (H0 Diterima): {nama_faktor} tidak memberikan pengaruh yang cukup nyata."

print("1.", ambil_keputusan(F_P, F_tabel_05, "Pendekatan Model (Perlakuan)"))
print("2.", ambil_keputusan(F_B, F_tabel_05, "Jumlah Iterasi (Baris)"))
print("3.", ambil_keputusan(F_K, F_tabel_05, "Ukuran Binning (Kolom)"))

# Cek model mana yang rata-ratanya paling kecil (paling baik)
rata_rata_model = df.groupby('nama_model')['gll_score'].mean().sort_values()
print("\nPERINGKAT MODEL BERDASARKAN RATA-RATA GLL LOSS (Lebih kecil lebih baik):")
for i, (model, skor) in enumerate(rata_rata_model.items(), 1):
    print(f"Peringkat {i}: {model} (Loss: {skor:.4f})")

Memproses Analisis Ragam (ANOVA) untuk Matriks RBSL 3x3...

--------------------------------------------------------------------------------
SUMBER KERAGAMAN     | DB  | JUMLAH KUADRAT (JK)  | KUADRAT TENGAH (KT)  | F-HITUNG   | P-VALUE
--------------------------------------------------------------------------------
Baris (Iterasi)      | 2   | -0.2500              | -0.1250              | -0.5000    | 1.0000
Kolom (Binning)      | 2   | -0.2500              | -0.1250              | -0.5000    | 1.0000
Perlakuan (Model)    | 2   | 1677537394173718.0000 | 838768697086859.0000 | 3355074788347436.0000 | 0.0000
Sisaan (Error)       | 2   | 0.5000               | 0.2500               | -          | -
--------------------------------------------------------------------------------
Total                | 8   | 1677537394173718.0000
--------------------------------------------------------------------------------

F-Tabel (alpha=0.05): 19.0000

KESIMPULAN PENELITIAN:
1. Signifikan (H1 Diterima)